Modelling of df_modelling_full.xlsx

# Libraries

In [59]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
import pandas as pd
from pathlib import Path
import sys

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [61]:
RANDOM_STATE = 42

In [62]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

# Splashing

## Train/test datasets creation

In [63]:
target = 'splashing'
train, test = get_train_test(
    dataset_filename='df_modelling_full',
    target=target)

## Numerical, categorical features selection

In [64]:
train.describe()

,splashing,wettability,roughness,liquid_density,surface_tension,viscosity,particle_mean_diameter,particle_density,droplet_diameter,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,velocity,Re,We,We_Re
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000
mean,0.646465,0.912458,2.068956,1116.161616,0.067515,0.015695,0.000083,1102.356902,0.003145,9.242424,0.117845,0.990754,0.727273,0.025974,3.641049,3437.867206,759.440848,171.713124
std,0.478874,0.825530,3.405158,98.337949,0.008979,0.009989,0.000080,320.569332,0.000271,15.987172,0.322969,0.269354,0.446113,0.024387,1.056598,4881.530866,403.820050,84.738011
min,0.000000,0.000000,0.040000,820.000000,0.026900,0.001040,0.000041,450.000000,0.002130,0.000000,0.000000,0.381356,0.000000,0.011339,1.980571,256.976895,142.008866,52.684981
25%,0.000000,0.000000,0.040000,1000.000000,0.067900,0.002050,0.000041,1000.000000,0.002990,0.000000,0.000000,0.877193,0.000000,0.012652,3.961141,600.961715,629.826251,128.988793
50%,1.000000,1.000000,0.100000,1180.000000,0.067900,0.023100,0.000041,1000.000000,0.003210,0.000000,0.000000,1.000000,1.000000,0.013577,3.961141,672.716015,807.132068,152.487586
75%,1.000000,2.000000,2.490000,1180.000000,0.071300,0.023100,0.000041,1200.000000,0.003340,20.000000,0.000000,1.016949,1.000000,0.018694,3.961141,6185.563594,903.933380,236.927284
max,1.000000,2.000000,10.890000,1180.000000,0.073200,0.023100,0.000275,2200.000000,0.003660,45.000000,1.000000,1.864407,1.000000,0.103774,5.941712,19139.168057,2036.917752,472.780129


In [65]:
for column in train.columns:
    print(f'{column}:\t|\t{[round(x, 3) for x in train[column].unique()[:5]]}\t|\tUnique vales: {train[column].nunique()}')

splashing:	|	[0, 1]	|	Unique vales: 2
wettability:	|	[1, 2, 0]	|	Unique vales: 3
roughness:	|	[0.04, 2.49, 10.89, 0.1]	|	Unique vales: 4
liquid_density:	|	[1180, 1000, 820, 1060, 1140]	|	Unique vales: 5
surface_tension:	|	[0.068, 0.073, 0.027, 0.071, 0.069]	|	Unique vales: 5
viscosity:	|	[0.023, 0.001, 0.007, 0.002, 0.007]	|	Unique vales: 5
particle_mean_diameter:	|	[0.0, 0.0, 0.0]	|	Unique vales: 3
particle_density:	|	[1000, 2200, 1200, 450]	|	Unique vales: 4
droplet_diameter:	|	[0.003, 0.003, 0.003, 0.003, 0.003]	|	Unique vales: 105
inclination:	|	[45, 0, 20]	|	Unique vales: 3
roughness_binary:	|	[0, 1]	|	Unique vales: 2
particle_liquid_density_ratio:	|	[0.847, 1.0, 1.864, 1.017, 1.22]	|	Unique vales: 8
volume_fraction_binary:	|	[1, 0]	|	Unique vales: 2
particle_droplet_diameter_ratio:	|	[0.014, 0.044, 0.014, 0.013, 0.012]	|	Unique vales: 148
velocity:	|	[1.981, 3.961, 5.942]	|	Unique vales: 3
Re:	|	[300.481, 11864.38, 16568.235, 6246.415, 682.304]	|	Unique vales: 196
We:	|	[202.465,

In [66]:
binary_features = ['roughness_binary', 'volume_fraction_binary']
categorical_features = ['wettability', 'roughness', 'liquid_density',
                        'surface_tension', 'viscosity', 'particle_mean_diameter',
                        'particle_density', 'inclination', 'velocity']
numerical_features = ['droplet_diameter', 'particle_droplet_diameter_ratio',
                      'particle_liquid_density_ratio', 'Re', 'We', 'We_Re']

In [67]:
assert len((set(train.columns)) & set(binary_features + categorical_features + numerical_features)) + 1 \
    == len(train.columns), 'Some features are not selected'

## Modelling

Models:
* LogReg
* SVM
* KNN
* CatBoost
* XGBoost

#### LogReg

In [6]:
from sklearn.linear_model import LogisticRegression

In [69]:
model = LogisticRegression()

In [70]:
features_to_leave = []
num_features = list(set(numerical_features) - set(features_to_leave))
cat_features = list(set(categorical_features) - set(features_to_leave))
transformers = []
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
transformers.append(
    ("num", numeric_transformer, num_features))
if categorical_features is not None:
    categorical_transformer = Pipeline(
        steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))])
    transformers.append(('cat', categorical_transformer, cat_features))
preprocessor = ColumnTransformer(transformers=transformers)
smt = SMOTE(random_state=RANDOM_STATE)
clf = Pipeline([("preprocessor", preprocessor), ("smt", smt), ("model", model)])

In [71]:
clf.fit(X=train.drop(columns=[target]), y=train[target])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['We', 'droplet_diameter',
                                                   'We_Re', 'Re',
                                                   'particle_liquid_density_ratio',
                                                   'particle_droplet_diameter_ratio']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['viscosity',
                                                   'particle_density',
                                                   'velocity', 'liquid_density',
                                                   'particle_mean_diameter',
                                                   'surface_tension',
                                                   'wettability', 'inclination',
                                                   'roughness'])])),
                ('smt', SMOTE(random_state=42)),
                ('model', LogisticRegression())])

In [73]:
preds = clf.predict(X=test.drop(columns=[target]))

In [101]:
from sklearn.svm import SVC

In [7]:
str(LogisticRegression().__class__).split('.')[-1][:-2]

'LogisticRegression'

In [10]:
class A:
    def __init__(self):
        self.a = 4
        self.b = ''
    def foo(self):
        self.b = 5

var = A()
var.b

4

In [93]:
dir(model)

['C',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_check_feature_names',
 '_check_n_features',
 '_estimator_type',
 '_get_param_names',
 '_get_tags',
 '_more_tags',
 '_predict_proba_lr',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_validate_data',
 'class_weight',
 'classes_',
 'coef_',
 'decision_function',
 'densify',
 'dual',
 'fit',
 'fit_intercept',
 'get_params',
 'intercept_',
 'intercept_scaling',
 'l1_ratio',
 'max_iter',
 'multi_class',
 'n_features_in_',
 'n_iter_',
 'n_jobs',
 'penalty',
 'predict',
 'predict_log_proba',
 'predict_proba',
 'random_state',
 'score',
 'set_params',
 'solver',
 'sparsify'

In [76]:
(test[target] == preds).sum() / test.shape[0]

0.8266666666666667

In [4]:
import uuid
uuid.uuid4().__str__().split('-')[0]

'57bc98ac'